In [ ]:
import json
import random
from pathlib import Path

data_dir = Path("../../data")
system_prompt = (Path("../../prompt/json_SYSTEM_prompt.txt")).read_text(encoding="utf-8")

rows = []
for user_file in sorted(data_dir.glob("user_prompt/*.txt")):
    name = user_file.stem  # e.g. "data1"
    report_file = data_dir / "report" / f"{name}.txt"
    json_file = data_dir / "json" / f"{name}.json"

    if not report_file.exists() or not json_file.exists():
        continue

    rows.append({
        "stem": name,
        "system_prompt": system_prompt,
        "user_prompt": user_file.read_text(encoding="utf-8"),
        "reasoning": report_file.read_text(encoding="utf-8"),
        "output": json_file.read_text(encoding="utf-8"),
    })

print(f"총 {len(rows)}개 샘플 로드 완료")

In [ ]:
msgs = []
for row in rows:
    assistant_response = f"{row['output']}"
    msgs.append({
        "stem": row["stem"],
        "messages": [
            {"role": "system", "content": row["system_prompt"]},
            {"role": "user", "content": row["user_prompt"]},
            {"role": "assistant", "content": assistant_response},
        ],
    })

# 90/10 분리 (seed 고정)
random.seed(42)
random.shuffle(msgs)

split = int(len(msgs) * 0.9)
train_msgs = msgs[:split]
test_msgs  = msgs[split:]

print(f"train: {len(train_msgs)}개 / test: {len(test_msgs)}개")

In [ ]:
# 학습용 저장 (stem 필드 제거)
output_path = Path("/LLaMA-Factory/data/custom-reasoning.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
train_data = [{"messages": m["messages"]} for m in train_msgs]
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
print(f"학습 데이터 저장: {output_path} ({len(train_data)}개)")

# 테스트용 stem 목록 저장
test_stems_path = data_dir / "test_stems.txt"
with open(test_stems_path, "w", encoding="utf-8") as f:
    f.write("\n".join(m["stem"] for m in test_msgs))
print(f"테스트 stem 목록 저장: {test_stems_path} ({len(test_msgs)}개)")

In [7]:
import json
from pathlib import Path

dataset_info_path = Path("/LLaMA-Factory/data/dataset_info.json")  # change if yours is elsewhere

new_dataset_name = "sunny_reasoning"
new_entry = {
    "file_name": "/LLaMA-Factory/data/custom-reasoning.json",
    "formatting": "sharegpt",
    "columns": {"messages": "messages"},
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant",
        "system_tag": "system",
    },
}

# Load existing dataset_info.json
with dataset_info_path.open("r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Add or update entry
dataset_info[new_dataset_name] = new_entry

# Save back
dataset_info_path.parent.mkdir(parents=True, exist_ok=True)
with dataset_info_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"Updated {dataset_info_path} with key: {new_dataset_name}")

Updated /LLaMA-Factory/data/dataset_info.json with key: sunny_reasoning
